# 4.2.3 CellChat to spatially resolved transcriptomics data

Inés Sentís

Date of execution

In [ ]:
Sys.Date()

In [ ]:
suppressMessages(suppressWarnings({
library(ggplot2)
library(Seurat)
library(dplyr)
library(tidyverse)
library(CellChat)
library(patchwork)
}))

## Parameters

In [ ]:
args <- commandArgs(TRUE)

pat <- args[2]
subproj <- args[3]
clustering <- args[4] # resolution choosen or niche

In [ ]:
source(here::here(glue::glue("{subproj}/st_analysis/misc/paths.R")))
set.seed(0)

## Load Data

In [ ]:
info_cohort <- read_csv(here::here("utils/info_samples.csv"),show_col_types = FALSE)
info_donor <- info_cohort %>% 
  dplyr::filter(patient == pat)

### Load spatial 10x visium

In [ ]:
rhab_tumor <- readRDS(here::here(glue::glue("{ann}/{robj_dir}/{pat}.rds")))

In [ ]:
gem_id <- info_donor %>%
  dplyr::filter(modality == "st") %>%
  pull(gem_id)

Make sure there are no rows with all 0

In [ ]:
keep_genes <- data.frame(rowSums(as.matrix(rhab_tumor[['Spatial']]@counts)) != 0)
colnames(keep_genes) <- "logical"
keep_genes <- subset(keep_genes, logical==TRUE)
rhab_tumor[['Spatial']]@counts <- rhab_tumor[['Spatial']]@counts[rownames(keep_genes), ]
rhab_tumor[['Spatial']]@data <- rhab_tumor[['Spatial']]@data[rownames(keep_genes), ]

Make sure there are no columns with all 0

In [ ]:
keep_spots <- data.frame(colSums(as.matrix(rhab_tumor[['Spatial']]@counts)) != 0)
colnames(keep_spots) <- "logical"
keep_spots <- subset(keep_spots, logical==TRUE)
rhab_tumor[['Spatial']]@counts <- rhab_tumor[['Spatial']]@counts[ ,rownames(keep_spots)]
rhab_tumor[['Spatial']]@data <- rhab_tumor[['Spatial']]@data[ ,rownames(keep_spots)]

In [ ]:
rhab_tumor <- NormalizeData(rhab_tumor, verbose = FALSE, assay = "Spatial")

### Niche or clustering

In [ ]:
if (clustering == "niche") {
    niches_data <- read_csv(here::here(glue::glue("AVGUSTINOVA_01_06_09_15/all_niches.csv")),
                       skip = 1, col_names = c("barcode", "sample", "niche"),show_col_types = FALSE)
    niches_data$niche <- as.character(niches_data$niche)
    
    niches_data <- niches_data %>%
        filter(sample == pat)
    
    df_meta <- rhab_tumor@meta.data %>%
        rownames_to_column("barcode")
    
    print(dim(df_meta))
    df_meta <- merge(df_meta, niches_data, by="barcode", all.x=T)
    print(dim(df_meta))
    
    rhab_tumor@meta.data$niche <- df_meta$niche
    
    default_colors_dict <- list('0'='#74add1',  
    '1'='#fee08b',  
    '2'='#5e4fa2',
    '3'='#abdda4',   
    '4'='#9e0142')
    }

In [ ]:
SpatialDimPlot(rhab_tumor, label = T, label.size = 3, group.by = clustering) +
    scale_fill_manual(breaks = names(default_colors_dict), values=unlist(default_colors_dict, use.names = F))

In [ ]:
table(niches_data$niche)

## Prepare CellChat input

In [ ]:
# get coordinates
spatial.locs = GetTissueCoordinates(rhab_tumor, scale = NULL, cols = c("imagerow", "imagecol"))

In [ ]:
# scale factors
scalefactors = jsonlite::fromJSON(txt = file.path(here::here(glue::glue("{spaceranger}/results/{gem_id}/outs/spatial")), 'scalefactors_json.json'))

In [ ]:
spot.size = 65 # the theoretical spot size (um) in 10X Visium
conversion.factor = spot.size/scalefactors$spot_diameter_fullres
spatial.factors = data.frame(ratio = conversion.factor, tol = spot.size/2)

d.spatial <- computeCellDistance(coordinates = spatial.locs, ratio = spatial.factors$ratio, tol = spatial.factors$tol)
min(d.spatial[d.spatial!=0]) # this value should approximately equal 100um for 10X Visium data

In [ ]:
data.input = rhab_tumor[['Spatial']]@data
meta = rhab_tumor@meta.data

meta$labels <- paste("N", as.character(meta[[clustering]]), sep="")

In [ ]:
# create object
cellchat <- createCellChat(object = data.input, meta = meta, group.by = "labels",
                           datatype = "spatial", coordinates = spatial.locs, spatial.factors = spatial.factors)

In [ ]:
names(default_colors_dict) <- paste0("N",names(default_colors_dict))

## Run CellChat

In [ ]:
CellChatDB <- CellChatDB.human

In [ ]:
CellChatDB.use <- subsetDB(CellChatDB, search = "Secreted Signaling", key = "annotation")
# set the used database in the object
cellchat@DB <- CellChatDB.use

In [ ]:
# subset the expression data of signaling genes for saving computation cost
cellchat <- subsetData(cellchat) # This step is necessary even if using the whole database
future::plan("multisession", workers = 4) 
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat, variable.both = F)

In [ ]:
#?computeCommunProb

In [ ]:
cellchat <- computeCommunProb(cellchat, type = "triMean",
                              population.size = TRUE,
                              distance.use = TRUE, 
                              interaction.range = 250,
                              scale.distance = 0.01,
                              contact.dependent = TRUE, 
                              contact.dependent.forced = TRUE, # contact spots between groups
                              contact.range = 250) 

In [ ]:
cellchat <- filterCommunication(cellchat, min.cells = 3)

In [ ]:
cellchat <- aggregateNet(cellchat)

In [ ]:
unlist(default_colors_dict)[unique(meta$labels)]

In [ ]:
colnames(cellchat@net$weight)

In [ ]:
plt1 <- netVisual_circle(cellchat@net$count, 
                         vertex.weight = rowSums(cellchat@net$count), 
                         weight.scale = T, label.edge= F, 
                         title.name = "Number of interactions", 
                         color.use = unlist(default_colors_dict)[colnames(cellchat@net$weight)]) 
plt2 <- netVisual_circle(cellchat@net$weight, 
                         vertex.weight = rowSums(cellchat@net$weight), 
                         weight.scale = T, label.edge= F, 
                         title.name = "Interaction weights/strength", 
                         color.use = unlist(default_colors_dict)[colnames(cellchat@net$weight)])

In [ ]:
netVisual_bubble(cellchat, 
                 remove.isolate = FALSE)

In [ ]:
# data frame consisting of all the inferred cell-cell communications at the level of ligands/receptors.
df.net <- subsetCommunication(cellchat)
head(df.net, n=10)

In [ ]:
df.net["clusters"] <- paste(df.net$source, df.net$target, sep=">")
df.net <- df.net %>%
    mutate(norm_prob = (prob - min(prob)) / (max(prob) - min(prob))) 

In [ ]:
write_csv(df.net, file = here::here(glue::glue("{ann}/{plt_dir}/4.2.3-cellchat_clust_{pat}.csv")))

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 15)
complot_tt <- ggplot(df.net, aes(x = clusters, y = interaction_name , size = norm_prob)) +
    geom_point(alpha = 0.7) +
    scale_size_continuous(range=c(1,10),   
                breaks = c(0, 0.25,0.50,0.75,1),
                labels = c("min","","","","max")) +
    facet_grid(pathway_name ~ ., scales = "free_y", space = "free_y", switch = "y") +
    theme_minimal() +
    theme(text = element_text(size = 16),
          plot.title = element_text(hjust = 0.5),
          axis.text.x = element_text(color = "black", angle = 45, hjust = 1),
          strip.text.y.left = element_text(angle = 0, hjust = 0, margin = margin(r = 4),face = "bold"),
          plot.margin = margin(10, 20, 10, 20)) +
    labs(title = pat, x = "clusters", 
         y = "L-R", size = "Communication Probability\n(strength given the sample)")
complot_tt

In [ ]:
if (dim(df.net)[1] > 200) {
    ggsave(complot_tt, file = here::here(glue::glue("{ann}/{plt_dir}/4.2.3-cellchat_clust_{pat}.svg")),
           width = 14, height = 20, dpi = 300)
    } else if(dim(df.net)[1] < 200 & dim(df.net)[1] > 80) {
    ggsave(complot_tt, file = here::here(glue::glue("{ann}/{plt_dir}/4.2.3-cellchat_clust_{pat}.svg")),
      width = 14, height =15, dpi = 300)
    } else {
    ggsave(complot_tt, file = here::here(glue::glue("{ann}/{plt_dir}/4.2.3-cellchat_clust_{pat}.svg")),
      width = 14, height =10, dpi = 300)
}

In [ ]:
# eye selected ones from most common ones
pathways.show <- c("BMP","WNT", "ncWNT", "SPP1", "NRG","HGF", "EGF", "SEMA3","FGF",
                   "MK", "GAS", "MIF", "TGFb", "PLAU","VEGF", "ANGPTL") 


list_plots <- lapply(pathways.show, function(x) {
    if (x %in% df.net$pathway_name) {
        print(x)
        p <- netVisual_aggregate(cellchat, signaling = x, layout = "spatial",
                                 sources.use = unique(meta$labels),targets.use =unique(meta$labels),
                                 color.use = unlist(default_colors_dict)[colnames(cellchat@net$weight)], 
                                 edge.width.max = 2, vertex.size.max = 1, alpha.image = 0.2, 
                                 vertex.label.cex = 3.5)
        p
    }
})


In [ ]:
list_plots <- Filter(Negate(is.null), list_plots)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
if (length(list_plots) !=  0) {
cp <- cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol = 4,
                   nrow = 2)
cp
}

In [ ]:
if (length(list_plots) !=  0){
    ggsave(plot = cp, width = 20, height = 10, dpi = 300, 
       file = here::here(glue::glue("{ann}/{plt_dir}/4.2.3-cellchat_pathway_{pat}.svg")))
}

In [ ]:
list_plots <- lapply(pathways.show, function(x) {
    if (x %in% df.net$pathway_name) {
        print(x)
        p <- netVisual_aggregate(cellchat, signaling = x, layout = "circle",
                                 sources.use = unique(meta$labels),targets.use =unique(meta$labels),
                                 color.use = unlist(default_colors_dict)[colnames(cellchat@net$weight)], 
                                 weight.scale = T)
        p
    }
})

In [ ]:
list_plots <- Filter(Negate(is.null), list_plots)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)
if (length(list_plots) !=  0) {
cp <- cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol = 3,
                   nrow = 3)
cp
}

In [ ]:
if (length(list_plots) !=  0){
ggsave(plot = cp, width = 20, height = 15, dpi = 300, 
       file = here::here(glue::glue("{ann}/{plt_dir}/4.2.3-cellchat_network_pathway_{pat}.svg")))
}

In [ ]:
top_pair <- df.net %>%
    arrange(desc(prob)) %>%
    slice_head(n = 10) %>%
    pull(interaction_name) %>%
    unique()

In [ ]:
list_plots <- lapply(as.vector(top_pair), function(x) {
    spatialFeaturePlot(cellchat, pairLR.use = x, point.size = 1, do.binary = TRUE, 
                   cutoff = 0.05, enriched.only = F, color.heatmap = "Reds", direction = 1)
})                 

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 10)
cp <- cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol = 3,
                   nrow = 2)
cp

## Save

In [ ]:
saveRDS(cellchat, file = here::here(glue::glue("{ann}/{robj_dir}/4.2.3-cellchat_clust_{pat}.rds")))

In [ ]:
sessionInfo()